In [1]:
import glob, sys, joblib, numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from torchvision import models, transforms as T
from transformers import AutoModel

NB = '/kaggle/input/notebooks/aadiparkhi'
A3, A4, A5 = [glob.glob(f'{NB}/{n}*')[0] for n in ('03a', '04', '05')]
CK = glob.glob('/kaggle/input/datasets/*/drift-multitask-checkpoints')[0]
torch.set_num_threads(4); rng = np.random.default_rng(0)

class M(nn.Module):
    def __init__(s, bb, d, dino=False):
        super().__init__()
        s.bb, s.dino = bb, dino
        s.h1, s.h2, s.h3 = nn.Linear(d, 48), nn.Linear(d, 11), nn.Linear(d, 2)
    def forward(s, x):
        if s.dino:
            h = s.bb(pixel_values=x).last_hidden_state
            z = torch.cat([h[:, 0], h[:, 1:].mean(1)], 1)
        else:
            z = s.bb(x)
        return z, s.h1(z), s.h2(z), s.h3(z)

cx = models.convnext_tiny(weights=None); cx.classifier[2] = nn.Identity()
nets = {'convnext': M(cx, 768), 'dino': M(AutoModel.from_pretrained('facebook/dinov2-small'), 768, True)}
for k, m in nets.items():
    m.load_state_dict(torch.load(f'{CK}/multitask_{k}.pt', map_location='cpu')); m.eval()

tf = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

@torch.no_grad()
def embed(imgs, bs=50):                      # imgs: PIL RGB images already at the domain's native size
    out = {k: {'emb': [], 'scene': [], 'cond': [], 'person': []} for k in nets}
    for i in range(0, len(imgs), bs):
        x = torch.stack([tf(im) for im in imgs[i:i + bs]])
        for k, m in nets.items():
            z, a, b, c = m(x)
            for n, v in zip(('emb', 'scene', 'cond', 'person'), (z, a, b, c)): out[k][n].append(v)
    return {k: {n: torch.cat(v).numpy() for n, v in d.items()} for k, d in out.items()}

meta = {d: pd.read_csv(f'{A3}/meta_{d}.csv') for d in ('places', 'bdd')}
feats = {d: np.load(f'{A3}/feats_{d}.npz') for d in meta}
PAR = {}
for d in meta:
    idx = rng.choice(np.where(meta[d].set == 'val')[0], 100, replace=False)
    r = embed([Image.open(meta[d].path[i]).convert('RGB') for i in idx]); PAR[d] = (idx, r)
    for k in nets:
        s, e = feats[d][f'{k}_emb'][idx].astype(np.float32), r[k]['emb']
        cos = (s * e).sum(1) / (np.linalg.norm(s, axis=1) * np.linalg.norm(e, axis=1))
        lg = feats[d][f'{k}_log'][idx]; mine = r[k]['scene'] if d == 'places' else r[k]['cond']
        agree = (lg[:, :mine.shape[1]].argmax(1) == mine.argmax(1)).mean()
        print(d, k, '| min cosine:', cos.min().round(5), '| mean rel diff:', (np.abs(s - e).mean() / np.abs(s).mean()).round(4), '| head argmax agreement:', agree)

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

places convnext | min cosine: 0.99999 | mean rel diff: 0.0012 | head argmax agreement: 1.0
places dino | min cosine: 1.0 | mean rel diff: 0.0017 | head argmax agreement: 0.99
bdd convnext | min cosine: 0.99998 | mean rel diff: 0.0022 | head argmax agreement: 1.0
bdd dino | min cosine: 1.0 | mean rel diff: 0.0021 | head argmax agreement: 1.0


In [2]:
sys.path.insert(0, A5); import drift_pipeline
mon = drift_pipeline.DriftMonitor(f'{A4}/detector_objects.joblib', f'{A5}/drift_artifacts.joblib')
BK = ['dino', 'convnext']

def compare(name, d, idx, r=None):
    if r is None: r = embed([Image.open(meta[d].path[i]).convert('RGB') for i in idx])
    st = mon.assess(d, {k: feats[d][f'{k}_emb'][idx].astype(np.float32) for k in BK}, feats[d]['dino_log'][idx] if d == 'bdd' else None)
    fr = mon.assess(d, {k: r[k]['emb'] for k in BK}, r['dino']['cond'] if d == 'bdd' else None)
    dv = np.abs(np.array(st['scores']) - np.array(fr['scores'])).max()
    print(f"{d:6s} {name:16s} stored: {st['level']:5s} {str(st['cause']):10s} {st['impact']:+.3f} | fresh: {fr['level']:5s} {str(fr['cause']):10s} {fr['impact']:+.3f} | max score diff {dv:.4f}")

for d in meta:
    compare('reference', d, *PAR[d])
    m_ = meta[d]; fog = np.where((m_.corruption == 'fog') & (m_.severity == 3))[0]
    compare('fog s3', d, rng.choice(fog, 100, replace=False))
mb = meta['bdd']; test = np.where(mb.set == 'test')[0]
twi = np.where((mb.set == 'test') & mb.cond_id.isin([2, 4, 7, 10]))[0]            # the four twilight conditions
compare('twilight mix 0.75', 'bdd', np.concatenate([rng.choice(twi, 75, replace=False), rng.choice(test, 25, replace=False)]))

places reference        stored: none  None       +0.000 | fresh: none  None       +0.000 | max score diff 0.0001
places fog s3           stored: act   fog        -0.145 | fresh: act   fog        -0.145 | max score diff 0.0001
bdd    reference        stored: none  None       +0.000 | fresh: none  None       +0.000 | max score diff 0.0001
bdd    fog s3           stored: act   fog        -0.214 | fresh: act   fog        -0.214 | max score diff 0.0003
bdd    twilight mix 0.75 stored: act   mix_shift  -0.179 | fresh: act   mix_shift  -0.172 | max score diff 0.0002


In [3]:
import os, shutil
OUT = '/kaggle/working/samples'; shutil.rmtree(OUT, ignore_errors=True)
NAMES = ['clear_day', 'clear_night', 'clear_twilight', 'overcast_day', 'overcast_twilight', 'rainy_day', 'rainy_night',
         'rainy_twilight', 'snow_fog_day', 'snow_fog_night', 'snow_fog_twilight']
GROUPS = {'night': lambda n: n.endswith('night'), 'twilight': lambda n: n.endswith('twilight'), 'rain': lambda n: n.startswith('rainy')}

def pick(d, mask, n): return rng.choice(np.where(mask)[0], n, replace=False)
def put(d, name, idx):
    os.makedirs(f'{OUT}/{d}/{name}')
    for j, i in enumerate(idx): shutil.copy(meta[d].path[i], f'{OUT}/{d}/{name}/{j:03d}{os.path.splitext(meta[d].path[i])[1]}')

pl, bd = meta['places'], meta['bdd']
put('places', 'reference', pick('places', pl.set == 'test', 100))
for c, s in (('fog', 3), ('defocus_blur', 3), ('brightness', 1)):
    put('places', f'{c}_s{s}', pick('places', (pl.corruption == c) & (pl.severity == s), 100))
put('bdd', 'reference', pick('bdd', bd.set == 'test', 100))
put('bdd', 'fog_s3', pick('bdd', (bd.corruption == 'fog') & (bd.severity == 3), 100))
for g, f in GROUPS.items():
    ids = [i for i, n in enumerate(NAMES) if f(n)]
    put('bdd', f'{g}_mix_75', np.concatenate([pick('bdd', (bd.set == 'test') & bd.cond_id.isin(ids), 75), pick('bdd', bd.set == 'test', 25)]))

rows = []
for d in ('places', 'bdd'):
    for name in sorted(os.listdir(f'{OUT}/{d}')):
        fs = sorted(os.listdir(f'{OUT}/{d}/{name}'))
        r = embed([Image.open(f'{OUT}/{d}/{name}/{f}').convert('RGB') for f in fs])
        a = mon.assess(d, {k: r[k]['emb'] for k in BK}, r['dino']['cond'] if d == 'bdd' else None)
        rows.append(dict(domain=d, folder=name, images=len(fs), level=a['level'], cause=a['cause'], impact=a['impact'], fired_by='+'.join(a['fired_by'])))
ex = pd.DataFrame(rows); ex.to_csv(f'{OUT}/EXPECTED.csv', index=False); print(ex.to_string(index=False))
for d in ('places', 'bdd'):
    f0 = sorted(os.listdir(f'{OUT}/{d}/reference'))[0]; print(d, 'native image size:', Image.open(f'{OUT}/{d}/reference/{f0}').size)
shutil.make_archive('/kaggle/working/samples', 'zip', OUT)
print('samples.zip:', round(os.path.getsize('/kaggle/working/samples.zip') / 1e6, 1), 'MB')

domain          folder  images level        cause  impact      fired_by
places   brightness_s1     100  none         None   0.000              
places defocus_blur_s3     100   act defocus_blur  -0.210 dino+convnext
places          fog_s3     100   act          fog  -0.165 dino+convnext
places       reference     100  none         None   0.000              
   bdd          fog_s3     100   act          fog  -0.220 dino+convnext
   bdd    night_mix_75     100 watch    mix_shift   0.070          dino
   bdd     rain_mix_75     100 watch    mix_shift   0.007          dino
   bdd       reference     100  none         None   0.000              
   bdd twilight_mix_75     100   act    mix_shift  -0.161 dino+convnext
places native image size: (224, 224)
bdd native image size: (384, 216)
samples.zip: 17.0 MB


In [4]:
import os, shutil
from transformers import AutoConfig
APP = '/kaggle/working/app'; os.makedirs(f'{APP}/models', exist_ok=True)

CORE = r"""# Embedding + drift analysis engine for the dashboard (CPU). Needs drift_pipeline.py next to this file.
import torch, torch.nn as nn
from PIL import Image
from torchvision import models, transforms as T
from transformers import AutoConfig, Dinov2Model
from drift_pipeline import DriftMonitor, BK

SIZE = {'places': (224, 224), 'bdd': (384, 216)}      # (width, height) of the images the detectors were fitted on
TF = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])


class MultiTask(nn.Module):
    def __init__(s, bb, d, dino=False):
        super().__init__()
        s.bb, s.dino = bb, dino
        s.h1, s.h2, s.h3 = nn.Linear(d, 48), nn.Linear(d, 11), nn.Linear(d, 2)

    def forward(s, x):
        if s.dino:
            h = s.bb(pixel_values=x).last_hidden_state
            z = torch.cat([h[:, 0], h[:, 1:].mean(1)], 1)
        else:
            z = s.bb(x)
        return z, s.h1(z), s.h2(z), s.h3(z)


def prepare(img, domain):
    # scale to cover the target size, then centre-crop (images already at the native size are untouched)
    img = img.convert('RGB'); w, h = SIZE[domain]
    if img.size == (w, h): return img
    s = max(w / img.width, h / img.height)
    img = img.resize((max(w, round(img.width * s)), max(h, round(img.height * s))), Image.BICUBIC)
    l, t = (img.width - w) // 2, (img.height - h) // 2
    return img.crop((l, t, l + w, t + h))


class Engine:
    def __init__(self, objects, artifacts, dino_pt, convnext_pt, dino_cfg):
        self.monitor = DriftMonitor(objects, artifacts)
        cx = models.convnext_tiny(weights=None); cx.classifier[2] = nn.Identity()
        self.nets = {'convnext': MultiTask(cx, 768),
                     'dino': MultiTask(Dinov2Model(AutoConfig.from_pretrained(str(dino_cfg))), 768, True)}
        for k, pt in (('convnext', convnext_pt), ('dino', dino_pt)):
            self.nets[k].load_state_dict(torch.load(pt, map_location='cpu')); self.nets[k].eval()

    @torch.no_grad()
    def embed(self, imgs, bs=25):
        out = {k: {'emb': [], 'cond': []} for k in self.nets}
        for i in range(0, len(imgs), bs):
            x = torch.stack([TF(im) for im in imgs[i:i + bs]])
            for k, m in self.nets.items():
                z, _, c, _ = m(x); out[k]['emb'].append(z); out[k]['cond'].append(c)
        return {k: {n: torch.cat(v).numpy() for n, v in d.items()} for k, d in out.items()}

    def analyze(self, imgs, domain):
        r = self.embed([prepare(i, domain) for i in imgs])
        return self.monitor.assess(domain, {k: r[k]['emb'] for k in BK}, r['dino']['cond'] if domain == 'bdd' else None)
"""

APPPY = r"""from pathlib import Path
import pandas as pd, streamlit as st
from PIL import Image
from core import Engine, SIZE

HERE = Path(__file__).parent; MD = HERE / 'models'
st.set_page_config(page_title='Drift monitor', layout='wide')


@st.cache_resource(show_spinner='Loading models (first run only)...')
def engine():
    return Engine(MD / 'detector_objects.joblib', MD / 'drift_artifacts.joblib',
                  MD / 'multitask_dino.pt', MD / 'multitask_convnext.pt', MD / 'dinov2_config')


DOMAINS = {'bdd': 'BDD100K driving scenes (time of day and weather)', 'places': 'Places365 scenes (48 classes)'}
BANNER = {'none': (st.success, 'No drift detected.'),
          'watch': (st.warning, 'Drift detected, but little or no accuracy loss is expected.'),
          'act': (st.error, 'Drift detected and the deployed model is likely to lose accuracy.')}


def why(cause):
    if cause == 'mix_shift':
        return ('Composition shift: the images look normal, but the mix of scenes has changed '
                '(for example more night or twilight scenes). The impact is estimated from the '
                'model\'s per-class accuracy.')
    return f'Image quality shift: {cause.replace("_", " ")}.'


st.title('Drift monitor: DINOv2 (deployed) with a ConvNeXt canary')
tab1, tab2 = st.tabs(['Live demo', 'About and results'])

with tab1:
    domain = st.radio('Domain', list(DOMAINS), format_func=DOMAINS.get, horizontal=True)
    w, h = SIZE[domain]
    st.caption(f'Images are scaled and centre-cropped to {w}x{h} before analysis. Best results with a batch of 100 images; '
               'under 30 images the alert is only indicative.')
    files = st.file_uploader('Upload a batch of images', type=['jpg', 'jpeg', 'png'], accept_multiple_files=True)
    if files:
        st.write(f'{len(files)} images selected.')
        if st.button('Analyze batch', type='primary'):
            imgs = [Image.open(f) for f in files]
            with st.spinner('Embedding images and checking for drift...'):
                res = engine().analyze(imgs, domain)
            show, msg = BANNER[res['level']]; show(msg)
            c1, c2, c3 = st.columns(3)
            c1.metric('Alert level', res['level'].upper())
            c2.metric('Estimated accuracy change', '-' if res['level'] == 'none' else f"{res['impact']:+.3f}")
            c3.metric('Flagged by', ' + '.join(res['fired_by']) or '-')
            if res['cause']: st.write(why(res['cause']))
            if res.get('note'): st.info(res['note'])
            with st.expander('Details'): st.json(res)
            st.image(imgs[:8], width=120)

with tab2:
    st.markdown('**How it works.** Two models embed each batch of images (DINOv2 is the deployed model, ConvNeXt is an '
                'independent canary). Each embedding is projected with PCA and scored with a K-S test and an autoencoder '
                'error. A batch is flagged when either sensor crosses its threshold (1% false-alarm budget). A random '
                'forest attributes the cause, and the expected accuracy change is estimated either by a loss regressor '
                '(image corruptions) or by a label-shift estimate (changes in the scene mix, BDD only). Alerts: none, '
                'watch (little expected loss) or act (expected loss of 5 points or more).')
    st.table(pd.DataFrame([
        ('False alarms on normal batches (1% budget)', '0.5-2% at 30, 50 and 100 images'),
        ('Cause attribution on corruptions (held out)', 'Places 96.1%, BDD 99.9%'),
        ('Accuracy-loss estimate, R2 (held out)', 'Places 0.88-0.89, BDD 0.83-0.84'),
        ('Scene-mix impact error (mean absolute)', '0.03-0.06 accuracy (corruption model alone: 0.05-0.12)'),
        ('Scene-mix shifts recognised as such (leave-one-group-out)', 'all six groups'),
    ], columns=['Result', 'Value']).set_index('Result'))
    st.markdown('**Known limitations.** Rain-heavy scene mixes are the weakest case (the estimate can come out as watch '
                'instead of act). The BDD test set is small, so expect roughly 1-4% false alarms on new BDD data. Impact '
                'models were trained on batches of 100 images, so estimates for other sizes are indicative. The scene-mix '
                'estimate exists only for BDD.')
    ex = HERE / 'samples' / 'EXPECTED.csv'
    if ex.exists():
        st.markdown('**Sample batches** (folders in `samples/`) and the alert each one gives:')
        st.dataframe(pd.read_csv(ex), hide_index=True)
"""

REQ = r"""streamlit
torch
torchvision
transformers
scikit-learn
scipy
pandas
numpy
pillow
joblib
"""

for name, txt in (('core.py', CORE), ('app.py', APPPY), ('requirements.txt', REQ)): open(f'{APP}/{name}', 'w').write(txt)
shutil.copy(f'{A5}/drift_pipeline.py', APP); shutil.copy(f'{A5}/versions.json', f'{APP}/models/')
shutil.copytree('/kaggle/working/samples', f'{APP}/samples', dirs_exist_ok=True)
AutoConfig.from_pretrained('facebook/dinov2-small').save_pretrained(f'{APP}/models/dinov2_config')    # offline architecture config
for f in (f'{A4}/detector_objects.joblib', f'{A5}/drift_artifacts.joblib', f'{CK}/multitask_dino.pt', f'{CK}/multitask_convnext.pt'):
    print(os.path.basename(f), round(os.path.getsize(f) / 1e6), 'MB')

sys.path.insert(0, APP); import core
eng = core.Engine(f'{A4}/detector_objects.joblib', f'{A5}/drift_artifacts.joblib', f'{CK}/multitask_dino.pt',
                  f'{CK}/multitask_convnext.pt', f'{APP}/models/dinov2_config')
SM = '/kaggle/working/samples'
def run(d, name, variant='', n=100, big=False):
    imgs = [Image.open(f'{SM}/{d}/{name}/{f}') for f in sorted(os.listdir(f'{SM}/{d}/{name}'))[:n]]
    if big: imgs = [im.resize((im.width * 3, im.height * 3), Image.BICUBIC) for im in imgs]        # simulate full-resolution uploads
    a = eng.analyze(imgs, d)
    print(f"{d:6s} {name:17s} {variant:12s} n={n:3d} -> {a['level']:5s} {str(a['cause']):10s} {a['impact']:+.3f} | {a.get('note', '')[:55]}")

run('bdd', 'reference'); run('bdd', 'twilight_mix_75'); run('bdd', 'twilight_mix_75', '3x resized', big=True)
run('places', 'reference'); run('places', 'fog_s3'); run('places', 'fog_s3', '3x resized', big=True)
run('bdd', 'reference', 'small batch', n=30); run('places', 'reference', 'small batch', n=30)

detector_objects.joblib 8 MB
drift_artifacts.joblib 380 MB
multitask_dino.pt 89 MB
multitask_convnext.pt 112 MB
bdd    reference                      n=100 -> none  None       +0.000 | 
bdd    twilight_mix_75                n=100 -> act   mix_shift  -0.161 | 
bdd    twilight_mix_75   3x resized   n=100 -> act   mix_shift  -0.169 | 
places reference                      n=100 -> none  None       +0.000 | 
places fog_s3                         n=100 -> act   fog        -0.165 | 
places fog_s3            3x resized   n=100 -> act   fog        -0.166 | 
bdd    reference         small batch  n= 30 -> none  None       +0.000 | impact models were trained on 100-image batches; treat 
places reference         small batch  n= 30 -> none  None       +0.000 | impact models were trained on 100-image batches; treat 


In [5]:
import json
ART = joblib.load(f'{A5}/drift_artifacts.joblib')
slim = dict(thr=ART['thr'], thr_by_B=ART['thr_by_B'], reg={dd: {'both': ART['reg'][dd]['both']} for dd in ART['reg']},
            att=ART['att'], C=ART['C'], acc_c=ART['acc_c'], pref=ART['pref'])
joblib.dump(slim, f'{APP}/models/drift_artifacts.joblib', compress=3)
shutil.copy(f'{A4}/detector_objects.joblib', f'{APP}/models/')
for k in ('dino', 'convnext'): shutil.copy(f'{CK}/multitask_{k}.pt', f'{APP}/models/')
print('slim artifacts:', round(os.path.getsize(f'{APP}/models/drift_artifacts.joblib') / 1e6), 'MB')

mon2 = drift_pipeline.DriftMonitor(f'{A4}/detector_objects.joblib', f'{APP}/models/drift_artifacts.joblib')
bad = 0
for d in meta:
    pool = np.where(meta[d].set.isin(['test', 'corrupt']))[0]
    for _ in range(15):
        b = rng.choice(pool, 100, replace=False)
        e = {k: feats[d][f'{k}_emb'][b].astype(np.float32) for k in BK}; lg = feats[d]['dino_log'][b] if d == 'bdd' else None
        bad += mon.assess(d, e, lg) != mon2.assess(d, e, lg)
print('slim vs full artifacts, differing results:', bad, 'of 30')

V = json.load(open(f'{APP}/models/versions.json')); print(V)
open(f'{APP}/requirements.txt', 'w').write(open(f'{APP}/requirements.txt').read().replace('scikit-learn', f"scikit-learn=={V['scikit_learn']}"))
open(f'{APP}/README.md', 'w').write('# Drift monitor dashboard\n\n1. Create a virtual environment and run: pip install -r requirements.txt '
    '(scikit-learn is pinned to the version used to train the forests).\n2. Run: streamlit run app.py\n3. Upload a batch of about 100 images. '
    'The samples/ folder has ready-made batches; EXPECTED.csv lists the alert each should give.\n')
shutil.make_archive('/kaggle/working/drift-dashboard', 'zip', '/kaggle/working', 'app')
print('drift-dashboard.zip:', round(os.path.getsize('/kaggle/working/drift-dashboard.zip') / 1e6), 'MB')

slim artifacts: 48 MB
slim vs full artifacts, differing results: 0 of 30
{'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.3.3', 'scipy': '1.16.3', 'scikit_learn': '1.6.1', 'torch': '2.11.0+cpu'}
drift-dashboard.zip: 257 MB
